In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer ,pipeline 
import torch
from pprint import pprint
import textwrap
from tqdm import tqdm

2025-11-06 22:58:15.481877: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-06 22:58:15.519927: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-06 22:58:16.830618: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [5]:
final_model = "./merged-model"

model = AutoModelForCausalLM.from_pretrained(final_model)
tokenizer = AutoTokenizer.from_pretrained(final_model)
pipe = pipeline('text-generation', model=model,tokenizer= tokenizer)

Device set to use cuda:0


In [10]:
len_wrap = 70
def generate(text):
    instruction = ("<|system|>"
                    "Please answer with one and only one of the option in the bracket" 
                    "That have been given by the user"
                    "And explain your choice."
                    "</s")
    tex = ("<|user|>" "Q : "+text+"</s>")
    tex = instruction + tex
    answer = pipe(tex, max_new_tokens=130, top_k=50, temperature=0.7, return_full_text=False)
    if not answer or answer[0]["generated_text"] =='':
        print("Bug detected please rerun ")
    l= textwrap.wrap(answer[0]['generated_text'].replace("'",""),len_wrap)
    final_ans = "\n".join(l)
    print(final_ans)    

In [11]:
text= ("I have a headache and I vomit a lot . I am 22 years old and I smoke a lot. What do you think I have")
generate(text)

 <|assistant|>A: Based on the given scenario, the patient is an adult
male with a history of smoking and headaches. The smoking history is
significant, and the headache is a common side effect of smoking. The
patients lifestyle factors and medical history have been properly
assessed and recorded. Therefore, the answer to the question is "a
smoker with a history of frequent headaches and vomiting". Since there
is no other information provided, it is concluded that this patient
has experienced frequent headaches and vomiting as a result of
smoking.


In [7]:
text = ( "Patient is a 45-year-old male with a history of hypertension and obesity. "
        "He presents with swelling in both lower legs and shortness of breath. "
        "These symptoms have worsened over the past 2 days. Given these risk factors and symptoms, "
        "the most concerning diagnosis is one of the following: "
        " { A) Congestive heart failure B) Deep vein thrombosis C) Pulmonary embolism}."
        )
generate(text)

 <|assistant|>Sure, based on the patients history, symptoms, and risk
factors, the most concerning diagnosis is moderate-severity congestive
heart failure. Congestive heart failure is a chronic condition
characterized by increased workload and decreased contractility of the
heart muscle. It can cause swelling in the legs and shortness of
breath due to fluid and blood pooling in the legs. The patients
symptoms have worsened over the past 2 days, which is consistent with
congestive heart failure.
